# Stage 1 (stability) — ML models, following Das et al. (ACS AMI 2024)

Targets: dissolution potential U_diss (as in Das et al.) and binding energy Ebin (physics-informed route).


Data: 196 TM@SxNy SACs (Xu 2025, exact Ebin) + 140 TM@g-CxNy SACs (Xu 2024, Ebin read from Fig. 2a, about ±0.18 eV).


Stability: Ef = Ebin + Ecoh (vs bulk metal);  U_diss = U0_diss − Ef/n_e (Das et al. eq. 3);  strict gate: Ebin<0, Ef<0, U_diss>0.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from mendeleev import element
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, LeaveOneGroupOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBRegressor
import shap
DATA='./'  # folder with the CSVs

## 1. Assemble the dataset

In [2]:
ref=pd.read_csv(DATA+'stability_reference_values.csv',index_col='Metal')
sn=pd.read_csv(DATA+'stage1_stability_SxNy_196SACs.csv')[['SAC','Metal','Support','Ebin']]; sn['Family']='SxNy'; sn['Ebin_source']='exact'
cn=pd.read_csv(DATA+'stage1_stability_CxNy_140SACs.csv')[['SAC','Metal','Support','Ebin_mid_eV']].rename(columns={'Ebin_mid_eV':'Ebin'}); cn['Family']='CxNy'; cn['Ebin_source']='digitized'
df=pd.concat([sn,cn],ignore_index=True).dropna(subset=['Ebin'])          # drops the 2 grey (positive, unread) C/N cells
df=df.join(ref[['Ecoh_eV','U0_diss_V','n_e']].astype(float),on='Metal')
df['Ef']=df.Ebin+df.Ecoh_eV; df['U_diss']=df.U0_diss_V-df.Ef/df.n_e
df['stable']=((df.Ebin<0)&(df.Ef.round(3)<0)&(df.U_diss.round(3)>0)).astype(int)   # rounded as in stage 1 (Ti@N4 sits at exactly 0.000 V)
# dopant (metal) features: cheap, tabulated
def nd(m):
    d=[(n,c) for (n,l),c in element(m).ec.conf.items() if l=='d']; return max(d)[1] if d else 0
M={}
for m in df.Metal.unique():
    e=element(m); M[m]=dict(Z=e.atomic_number, period=e.period, group=e.group_id, Nd=nd(m), chi_M=e.en_pauling,
                            IE1=e.ionenergies.get(1), EA=(e.electron_affinity or 0.0), r_atom=e.atomic_radius)
df=df.join(pd.DataFrame(M).T.astype(float),on='Metal')
# site / support features
CHI={'C':2.55,'N':3.04,'S':2.58}
def site(s):
    if s.startswith('g-'):                                  # g-CxNy: substrate composition (TM coordination not tabulated)
        import re; a,b=re.findall(r'C(\d+)N(\d*)',s)[0]; x,y=int(a),int(b or 1)
        return dict(n_S=0, N_frac=y/(x+y), chi_site=(x*CHI['C']+y*CHI['N'])/(x+y), is_CN=1)
    nS=int(s[1]) if s.startswith('S') else 0               # SxNy: the 4 atoms bonded to the metal
    return dict(n_S=nS, N_frac=(4-nS)/4, chi_site=(nS*CHI['S']+(4-nS)*CHI['N'])/4, is_CN=0)
df=df.join(pd.DataFrame({s:site(s) for s in df.Support.unique()}).T.astype(float),on='Support')
df['EN_diff']=df.chi_site-df.chi_M
onehot=pd.get_dummies(df.Support,prefix='sup').astype(int); df=pd.concat([df,onehot],axis=1)
F_metal=['Z','period','group','Nd','chi_M','IE1','EA','r_atom','Ecoh_eV','U0_diss_V','n_e']
F_site=['n_S','N_frac','chi_site','is_CN','EN_diff']
FEATS={'intrinsic (metal + site descriptors)':F_metal+F_site, 'intrinsic + support one-hot':F_metal+F_site+list(onehot.columns)}
print(df.groupby('Family').size().to_dict(),'| rows with U_diss:',df.U_diss.notna().sum(),'| stable:',df.stable.sum())

{'CxNy': 138, 'SxNy': 196} | rows with U_diss: 324 | stable: 57


## 2. Models (Das et al.: KRR, RFR, GBR, XGBR; 80:20 split, grid-search tuning, 5-fold CV)

In [3]:
MODELS={
 'KRR':(make_pipeline(StandardScaler(),KernelRidge()),{'kernelridge__kernel':['laplacian','rbf'],'kernelridge__alpha':[1e-3,1e-2,1e-1,1],'kernelridge__gamma':[0.005,0.01,0.05,0.1]}),
 'RFR':(RandomForestRegressor(n_estimators=400,random_state=0),{'max_depth':[None,8],'min_samples_leaf':[1,2],'max_features':['sqrt',0.5,1.0]}),
 'GBR':(GradientBoostingRegressor(random_state=0),{'n_estimators':[300,600],'learning_rate':[0.03,0.1],'max_depth':[2,3]}),
 'XGBR':(XGBRegressor(random_state=0,verbosity=0,n_jobs=2),{'n_estimators':[300,600],'learning_rate':[0.03,0.1],'max_depth':[2,3,4]}),
}
def fit_eval(d,target,feats):
    X=d[feats].values; y=d[target].values
    Xtr,Xte,ytr,yte,itr,ite=train_test_split(X,y,np.arange(len(d)),test_size=0.2,random_state=42)
    rows=[];best={}
    for name,(est,grid) in MODELS.items():
        gs=GridSearchCV(est,grid,cv=KFold(5,shuffle=True,random_state=0),scoring='neg_mean_absolute_error',n_jobs=1).fit(Xtr,ytr)
        m=gs.best_estimator_
        cvp=cross_val_predict(m,X,y,cv=KFold(5,shuffle=True,random_state=1))
        rows.append(dict(model=name,train_MAE=mean_absolute_error(ytr,m.predict(Xtr)),test_MAE=mean_absolute_error(yte,m.predict(Xte)),
                         test_R2=r2_score(yte,m.predict(Xte)),CV5_MAE=mean_absolute_error(y,cvp),CV5_R2=r2_score(y,cvp)))
        best[name]=(m,gs.best_params_,cvp,ite)
    return pd.DataFrame(rows).round(3),best

## 3. Target A — U_diss directly (Das et al. approach)

In [4]:
dU=df.dropna(subset=['U_diss']).reset_index(drop=True)
resU={}
for fs,feats in FEATS.items():
    t,b=fit_eval(dU,'U_diss',feats); resU[fs]=(t,b); print('U_diss |',fs); print(t.to_string(index=False),'\n')

U_diss | intrinsic (metal + site descriptors)
model  train_MAE  test_MAE  test_R2  CV5_MAE  CV5_R2
  KRR      0.061     0.179    0.934    0.165   0.938
  RFR      0.109     0.264    0.864    0.267   0.845
  GBR      0.095     0.246    0.893    0.207   0.912
 XGBR      0.101     0.241    0.891    0.200   0.915 



U_diss | intrinsic + support one-hot
model  train_MAE  test_MAE  test_R2  CV5_MAE  CV5_R2
  KRR      0.021     0.190    0.934    0.164   0.943
  RFR      0.078     0.247    0.885    0.225   0.896
  GBR      0.013     0.240    0.895    0.190   0.927
 XGBR      0.023     0.230    0.912    0.192   0.928 



## 4. Target B — Ebin, then U_diss from the formula (physics-informed)

In [5]:
dE=df.dropna(subset=['U_diss']).reset_index(drop=True)   # same 324 SACs as the U_diss route (Hg has no cohesive energy)
resE={}
for fs,feats in FEATS.items():
    t,b=fit_eval(dE,'Ebin',feats); resE[fs]=(t,b); print('Ebin |',fs); print(t.to_string(index=False),'\n')

Ebin | intrinsic (metal + site descriptors)
model  train_MAE  test_MAE  test_R2  CV5_MAE  CV5_R2
  KRR      0.142     0.358    0.955    0.377   0.955
  RFR      0.267     0.598    0.895    0.594   0.883
  GBR      0.212     0.505    0.918    0.466   0.932
 XGBR      0.231     0.492    0.925    0.470   0.931 



Ebin | intrinsic + support one-hot
model  train_MAE  test_MAE  test_R2  CV5_MAE  CV5_R2
  KRR      0.045     0.367    0.959    0.343   0.963
  RFR      0.179     0.492    0.922    0.476   0.929
  GBR      0.035     0.455    0.937    0.428   0.945
 XGBR      0.059     0.446    0.941    0.414   0.950 



## 5. Does the model reproduce the stability verdict? (5-fold CV predictions, strict gate)

In [6]:
def gate_scores(pred_ebin=None,pred_u=None,d=None):
    if pred_ebin is not None:
        ef=pred_ebin+d.Ecoh_eV.values; u=d.U0_diss_V.values-ef/d.n_e.values; st=((pred_ebin<0)&(ef<0)&(u>0)).astype(int); uerr=np.abs(u-d.U_diss.values)
    else:
        u=pred_u; ef=(d.U0_diss_V.values-u)*d.n_e.values; st=((d.Ebin.values<0)&(ef<0)&(u>0)).astype(int); uerr=np.abs(u-d.U_diss.values)
    yt=d.stable.values
    return dict(U_diss_MAE=np.nanmean(uerr),accuracy=accuracy_score(yt,st),precision=precision_score(yt,st),recall=recall_score(yt,st),F1=f1_score(yt,st))
rows=[]
for fs in FEATS:
    for name in MODELS:
        cvpE=resE[fs][1][name][2]; mask=dE.U_diss.notna().values
        rows.append(dict(features=fs,model=name,route='Ebin -> formula',**gate_scores(pred_ebin=cvpE[mask],d=dE[mask].reset_index(drop=True))))
        rows.append(dict(features=fs,model=name,route='U_diss direct',**gate_scores(pred_u=resU[fs][1][name][2],d=dU)))
G=pd.DataFrame(rows).round(3); print(G.sort_values('F1',ascending=False).to_string(index=False))

                            features model           route  U_diss_MAE  accuracy  precision  recall    F1
         intrinsic + support one-hot   KRR   U_diss direct       0.164     0.948      0.870   0.825 0.847
intrinsic (metal + site descriptors)   KRR   U_diss direct       0.165     0.944      0.868   0.807 0.836
         intrinsic + support one-hot   GBR   U_diss direct       0.190     0.944      0.868   0.807 0.836
         intrinsic + support one-hot   KRR Ebin -> formula       0.146     0.944      0.882   0.789 0.833
intrinsic (metal + site descriptors)   KRR Ebin -> formula       0.158     0.941      0.896   0.754 0.819
         intrinsic + support one-hot  XGBR   U_diss direct       0.192     0.932      0.818   0.789 0.804
intrinsic (metal + site descriptors)  XGBR   U_diss direct       0.200     0.932      0.843   0.754 0.796
intrinsic (metal + site descriptors)   GBR   U_diss direct       0.207     0.929      0.827   0.754 0.789
         intrinsic + support one-hot   GBR Ebi

## 6. Transferability: leave one support out, leave one metal out (Ebin route, intrinsic features)

In [7]:
def logo(target,feats,groups,name):
    m,params,_,_=resE['intrinsic (metal + site descriptors)'][1][name]
    p=cross_val_predict(m,dE[feats].values,dE[target].values,cv=LeaveOneGroupOut(),groups=groups)
    return p
fi=FEATS['intrinsic (metal + site descriptors)']
T=[]
for name in ['KRR','XGBR']:
    for gname,g in [('support',dE.Support.values),('metal',dE.Metal.values)]:
        p=logo('Ebin',fi,g,name); err=np.abs(p-dE.Ebin.values)
        T.append(dict(model=name,left_out=gname,MAE=err.mean(),R2=r2_score(dE.Ebin,p)))
        if gname=='support': print(name,'LOSO MAE by support:',pd.Series(err).groupby(dE.Support.values).mean().round(2).to_dict())
print(pd.DataFrame(T).round(3).to_string(index=False))

KRR LOSO MAE by support: {'N4': 2.07, 'S1N3': 0.37, 'S2N2-HEX': 0.56, 'S2N2-OPP': 0.68, 'S2N2-PEN': 0.4, 'S3N1': 0.6, 'S4': 0.6, 'g-C2N': 1.74, 'g-C3N4': 0.6, 'g-C4N3': 3.07, 'g-C6N6': 0.85, 'g-C9N10': 0.45}


XGBR LOSO MAE by support: {'N4': 1.76, 'S1N3': 0.68, 'S2N2-HEX': 0.54, 'S2N2-OPP': 0.66, 'S2N2-PEN': 0.38, 'S3N1': 0.63, 'S4': 0.57, 'g-C2N': 2.75, 'g-C3N4': 1.13, 'g-C4N3': 2.92, 'g-C6N6': 0.6, 'g-C9N10': 0.45}


model left_out   MAE    R2
  KRR  support 0.999 0.643
  KRR    metal 0.840 0.797
 XGBR  support 1.088 0.574
 XGBR    metal 0.923 0.745


## 7. SHAP — which features control stability? (best tree model on Ebin, intrinsic features)

In [8]:
m=resE['intrinsic (metal + site descriptors)'][1]['XGBR'][0].fit(dE[fi].values,dE.Ebin.values)
sv=shap.TreeExplainer(m).shap_values(dE[fi].values)
imp=pd.Series(np.abs(sv).mean(axis=0),index=fi).sort_values(ascending=False); print(imp.round(3).to_string())
fig,ax=plt.subplots(figsize=(6,4)); imp[::-1].plot.barh(ax=ax,color='#4c72b0'); ax.set_xlabel('mean |SHAP| (eV)'); ax.set_title('What sets Ebin (XGBR, 336 SACs)'); plt.tight_layout(); plt.savefig(DATA+'stability_shap_Ebin.png',dpi=200)
# parity plot for the best direct-U_diss and Ebin-route models (CV predictions)
fig,axs=plt.subplots(1,2,figsize=(9,4))
for ax,(lab,d,p) in zip(axs,[('Ebin (eV)',dE,resE['intrinsic + support one-hot'][1]['KRR'][2]),('U_diss (V)',dU,resU['intrinsic + support one-hot'][1]['KRR'][2])]):
    y=(d.Ebin if lab.startswith('E') else d.U_diss).values
    for fam,col,name in [('SxNy','#4c72b0','S/N-graphene (exact DFT)'),('CxNy','#dd8452','carbon nitride (DFT read from heatmap)')]:
        k=(d.Family==fam).values; ax.scatter(y[k],p[k],s=10,c=col,label=name)
    lo,hi=min(y.min(),p.min()),max(y.max(),p.max()); ax.plot([lo,hi],[lo,hi],'k--',lw=1)
    ax.set_xlabel('DFT '+lab); ax.set_ylabel('KRR predicted (5-fold CV)'); ax.set_title(f'{lab}: MAE {mean_absolute_error(y,p):.2f}, R² {r2_score(y,p):.2f}')
axs[0].legend(fontsize=8,loc='upper left')
plt.tight_layout(); plt.savefig(DATA+'stability_parity_KRR.png',dpi=200)

Ecoh_eV      0.649
N_frac       0.548
Nd           0.388
EN_diff      0.381
group        0.362
n_S          0.296
is_CN        0.214
chi_M        0.140
IE1          0.105
EA           0.090
Z            0.078
U0_diss_V    0.036
r_atom       0.028
chi_site     0.009
n_e          0.006
period       0.000


## 8. Save CV predictions

In [9]:
out=dE[['SAC','Metal','Support','Family','Ebin_source','Ebin','Ef','U_diss','stable']].copy()
out['Ebin_pred_KRR_CV']=resE['intrinsic + support one-hot'][1]['KRR'][2]
out['Ebin_pred_XGBR_CV']=resE['intrinsic + support one-hot'][1]['XGBR'][2]
out.round(3).to_csv(DATA+'stability_ml_predictions.csv',index=False); print('saved')

saved
